### CineBot: A movie ticket Booking Assistant

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

True

In [8]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [18]:
model = ChatOpenAI(model="gpt-5-nano")

In [19]:
model.profile

{'name': 'GPT-5 Nano',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True}

In [15]:
model.profile['structured_output']

True

## Topics
* Structured Outputs
    * Show how AI doesnot follow same structure on all requests
    * How with_structured_output() helps in this using BookingRequest Pydantic model
    * 

* Provider Strategy & Tool Strategy
    * Provider Strategy (default) with_structured_output, always use when supported by model.
        * with create_agent() call, pass Pydantic modle in response_format parameter.
        * With Model invocation, use with_structured_output.
    * Tool Strategy: if provider deosn't support with_structured_output, but tool calling is supported.
    * Mermaid diagram showing structured strategy (in provided notbooks).
    * We can provide multiple Schema/Pydantic models to response_format, and let model decide


<img src="../../assets/Screenshot 2026-07-25 at 8.25.07 AM.png" width="800" height="500">


In [5]:
booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "can u book me a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT - need to CANCEL my booking for Oppenheimer, confirmation was under Aisha",
]


In [ ]:
for msg in booking_requests:
    r = model.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")


In [11]:
class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether this is a new booking or a cancellation")
    ticket_count: int = Field(description="How many tickets, default 1 if not mentioned", default=1)

structured_model = model.with_structured_output(BookingRequest)

In [12]:
for msg in booking_requests:
    result = structured_model.invoke(f"Extract b booking request from: {msg}")
    print(result)
    print(f" --> action type : {type(result.action)}, value : {result.action}")
    print("---")


customer_name='Priya' movie_title='Interstellar' action='book' ticket_count=2
 --> action type : <class 'str'>, value : book
---
customer_name='Rohan' movie_title='Dune Part Two' action='book' ticket_count=1
 --> action type : <class 'str'>, value : book
---
customer_name='Aisha' movie_title='Oppenheimer' action='cancel' ticket_count=1
 --> action type : <class 'str'>, value : cancel
---
